# Step 5 — CPU vs GPU Runtime Comparison

Author(s): Brandon Antone

This notebook reads the runtimes recorded by `02_etl_v3.py` (CPU) and `04_spark_rapids_etl.py` (GPU) and visualizes the speedup achieved with Spark RAPIDS.

**Before running this notebook**, make sure you have run, in order:
1. `02_etl_v3.py` — CPU-only run of the ETL job
2. `04_spark_rapids_etl.py` — same ETL logic with Spark RAPIDS / GPU enabled

Both scripts write their elapsed wall-clock time to `runtime_metrics.json` in this directory.

In [ ]:
import json
import os

import matplotlib.pyplot as plt

RUNTIME_METRICS_PATH = "/home/cdsw/spark-rapids-qualification-tool/runtime_metrics.json"

if not os.path.exists(RUNTIME_METRICS_PATH):
    raise FileNotFoundError(
        "runtime_metrics.json not found. Run 02_etl_v3.py and 04_spark_rapids_etl.py first."
    )

with open(RUNTIME_METRICS_PATH, "r") as f:
    metrics = json.load(f)

missing = [mode for mode in ("cpu", "gpu") if mode not in metrics]
if missing:
    raise KeyError(
        f"Missing runtime entries for: {missing}. "
        "Run the corresponding ETL script(s) before comparing."
    )

metrics

In [ ]:
cpu_seconds = metrics["cpu"]["seconds"]
gpu_seconds = metrics["gpu"]["seconds"]
speedup = cpu_seconds / gpu_seconds

print(f"CPU runtime: {cpu_seconds:.2f}s")
print(f"GPU runtime: {gpu_seconds:.2f}s")
print(f"Speedup: {speedup:.2f}x")

In [ ]:
labels = ["CPU", "GPU (Spark RAPIDS)"]
values = [cpu_seconds, gpu_seconds]
colors = ["#5b7db1", "#76b041"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, values, color=colors)

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1f}s",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )

ax.set_ylabel("Runtime (seconds)")
ax.set_title(f"ETL v3 Runtime: CPU vs GPU  ({speedup:.2f}x speedup)")

plt.tight_layout()
plt.show()